# 🔐 AI-Powered GitHub Security Vulnerability Scanner
### Using Hugging Face `google/flan-t5-large` · OSV API · Excel Reporting

---

## 📘 1. Introduction

This notebook scans the GitHub repository **`mayureshnadkar-beep/spring-ai-ollama`** for security vulnerabilities and uses a **Hugging Face AI model** to explain each finding, classify severity, assess risk impact, and provide remediation with secure code examples.

### What this scanner does:

| Stage | Description |
|---|---|
| 📥 Fetch | Recursively retrieve `.java`, `.properties`, `.xml`, `.yml`, `.env` files |
| 🔍 Detect | 20+ regex patterns — secrets, Java risks, SQL injection, weak crypto |
| 📦 Dependencies | Parse `pom.xml` and query the free OSV vulnerability database |
| 🤖 AI Analysis | `google/flan-t5-large` via Hugging Face Inference API |
| 📊 Score | Numeric risk score → Safe / Moderate / High Risk / Critical Risk |
| 📈 Visualize | Bar chart, pie chart, risk gauge |
| 📄 Export | Professional 2-sheet Excel report |

### Architecture
```
GitHub API ──► File Parser ──► Vulnerability Engine ──► HuggingFace AI
                                      │                        │
                               OSV Dep Check            Explanations
                                      │                  Severity
                                      └──── Risk Scorer ──► Excel + Charts
```

### Authentication Required
- **GitHub Token** — optional but recommended (avoids 60 req/hr limit)  
  Get one free at: https://github.com/settings/tokens → _repo (read)_ scope
- **Hugging Face Token** — free at: https://huggingface.co/settings/tokens  
  Required for Inference API calls to `google/flan-t5-large`

> ⚠️ **No passwords are used.** GitHub removed password-based API auth in August 2021.

---
## 📦 2. Install Dependencies

In [4]:
import subprocess, sys

PACKAGES = [
    'requests', 'pandas', 'openpyxl', 'PyGithub',
    'matplotlib', 'seaborn', 'tqdm',
    'transformers', 'torch', 'huggingface_hub', 'lxml'
]

print('Installing packages...\n')
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q', '--upgrade'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status}  {pkg}')

print('\n🎉 All dependencies ready!')

Installing packages...

  ✅  requests
  ✅  pandas
  ✅  openpyxl
  ✅  PyGithub
  ✅  matplotlib
  ✅  seaborn
  ✅  tqdm
  ✅  transformers
  ✅  torch
  ✅  huggingface_hub
  ✅  lxml

🎉 All dependencies ready!


---
## ⚙️ 3. Imports & Global Configuration

In [19]:
import re, os, json, time, base64, warnings, textwrap
from datetime import datetime
from xml.etree import ElementTree as ET

import requests
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from github import Github, GithubException
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans'})

# ── Severity mapping ────────────────────────────────────────────────
SEVERITY_SCORE   = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
SEVERITY_COLOR   = {
    'Low'     : '#2ECC71',
    'Medium'  : '#F39C12',
    'High'    : '#E74C3C',
    'Critical': '#8E44AD'
}
SEVERITY_BG_HEX  = {
    'Low'     : 'EAFAF1',
    'Medium'  : 'FEF9E7',
    'High'    : 'FDEDEC',
    'Critical': 'F5EEF8'
}
SEVERITY_FG_HEX  = {
    'Low'     : '1E8449',
    'Medium'  : 'B7770D',
    'High'    : 'C0392B',
    'Critical': '6C3483'
}

# ── Risk thresholds ─────────────────────────────────────────────────
RISK_BANDS = [
    (5,  'Safe',          '#2ECC71'),
    (15, 'Moderate Risk', '#F39C12'),
    (30, 'High Risk',     '#E74C3C'),
    (999,'Critical Risk', '#8E44AD'),
]

# ── Constants ───────────────────────────────────────────────────────
OSV_API          = 'https://api.osv.dev/v1/query'
HF_API_BASE      = 'https://api-inference.huggingface.co/models'
HF_MODEL         = 'google/flan-t5-large'
OUTPUT_XLSX      = 'github_security_vulnerability_report.xlsx'
TARGET_REPO      = 'https://github.com/mayureshnadkar-beep/spring-ai-ollama'
MAX_FILES        = 120
HF_MAX_TOKENS    = 400
HF_CALL_DELAY    = 1.5   # seconds between HF calls

SCAN_EXTENSIONS  = {'.java', '.properties', '.yml', '.yaml', '.xml', '.env', '.conf', '.gradle'}
DEP_FILES        = {'pom.xml', 'build.gradle', 'build.gradle.kts'}

print('✅ Configuration loaded.')
print(f'   Target repo   : {TARGET_REPO}')
print(f'   HF Model      : {HF_MODEL}')
print(f'   Output file   : {OUTPUT_XLSX}')

✅ Configuration loaded.
   Target repo   : https://github.com/mayureshnadkar-beep/spring-ai-ollama
   HF Model      : google/flan-t5-large
   Output file   : github_security_vulnerability_report.xlsx


---
## 🔑 4. GitHub & Hugging Face Setup

Fill in your tokens below. Both are **free** and take < 2 minutes to obtain.

- **GitHub token**: https://github.com/settings/tokens → _New classic token_ → tick `repo`
- **HuggingFace token**: https://huggingface.co/settings/tokens → _New token_ → _Read_

In [21]:
# ══════════════════════════════════════════════════════
#  ✏️  PASTE YOUR TOKENS HERE
# ══════════════════════════════════════════════════════

GITHUB_TOKEN = ''          # Optional — leave blank for unauthenticated (60 req/hr)
HF_TOKEN     = ''          # Required for HF Inference API  (hf_xxxxx...)

# ══════════════════════════════════════════════════════

# ── GitHub client ────────────────────────────────────
from github import Github

def make_github_client(token: str = ""):
    if token.strip():
        g = Github(token.strip())
        user = g.get_user().login

        try:
            rate = g.get_rate_limit()
            print(f"✅ GitHub authenticated as '{user}'")
            print(f"Rate Limit: {rate}")
        except Exception:
            pass
    else:
        g = Github()
        print("⚠️ GitHub unauthenticated")

    return g

# ── HuggingFace headers ──────────────────────────────
def make_hf_headers(token: str) -> dict:
    if token.strip():
        print(f'✅ HuggingFace  →  token configured ({token[:8]}...)')
        return {'Authorization': f'Bearer {token.strip()}'}
    else:
        print('⚠️  HuggingFace  →  no token. AI calls will use rule-based fallback.')
        return {}

github_client = make_github_client(GITHUB_TOKEN)
hf_headers    = make_hf_headers(HF_TOKEN)

✅ GitHub authenticated as 'mayureshnadkar-beep'
Rate Limit: RateLimitOverview(rate=Rate(reset=2026-06-10 13:21:23+00:00, remaining=4965, limit=5000))
✅ HuggingFace  →  token configured (hf_ypzAt...)


---
## 📥 5. Fetch Repository Files

In [23]:
def decode_content(item) -> str:
    """Safely base64-decode a GitHub file's content."""
    try:
        if item.encoding == 'base64':
            return base64.b64decode(item.content).decode('utf-8', errors='replace')
        return item.content or ''
    except Exception:
        return ''


def fetch_repo_files(g: Github, repo_name: str) -> dict:
    """
    Recursively walk the repository and collect:
      - source code files matching SCAN_EXTENSIONS
      - dependency files (pom.xml, build.gradle)
    Returns a dict with keys: repo_meta, code_files, dep_files, repo_obj
    """
    print(f'\n📥 Connecting to: {repo_name}')
    try:
        repo = g.get_repo(repo_name)
    except GithubException as exc:
        msg = exc.data.get('message', str(exc)) if hasattr(exc, 'data') else str(exc)
        raise RuntimeError(f'Cannot access "{repo_name}": {msg}')

    meta = {
        'name'       : repo.name,
        'full_name'  : repo.full_name,
        'url'        : repo.html_url,
        'language'   : repo.language or 'N/A',
        'description': repo.description or 'N/A',
        'stars'      : repo.stargazers_count,
        'forks'      : repo.forks_count,
        'size_kb'    : repo.size,
        'scanned_at' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
    print(f'   ✅ {repo.full_name}  |  ⭐ {repo.stargazers_count}  |  Lang: {repo.language}')

    code_files, dep_files = [], []
    total = [0]

    def walk(contents, depth=0):
        if total[0] >= MAX_FILES:
            return
        for item in contents:
            if total[0] >= MAX_FILES:
                break
            if item.type == 'dir' and depth < 6:
                try:
                    walk(repo.get_contents(item.path), depth + 1)
                except GithubException:
                    pass
            elif item.type == 'file':
                name_lower = item.name.lower()
                ext        = os.path.splitext(name_lower)[1]
                if name_lower in DEP_FILES:
                    dep_files.append(item)
                    total[0] += 1
                elif ext in SCAN_EXTENSIONS and item.size < 400_000:
                    code_files.append(item)
                    total[0] += 1

    try:
        walk(repo.get_contents(''))
    except GithubException as exc:
        print(f'   ⚠️  Partial traversal: {exc}')

    print(f'\n   📄 Source files   : {len(code_files)}')
    print(f'   📦 Dep files      : {len(dep_files)}')
    print(f'   📊 Total fetched  : {total[0]} (cap={MAX_FILES})')

    return {
        'repo_meta' : meta,
        'code_files': code_files,
        'dep_files' : dep_files,
        'repo_obj'  : repo,
    }


repo_data = fetch_repo_files(github_client, TARGET_REPO)


📥 Connecting to: https://github.com/mayureshnadkar-beep/spring-ai-ollama


RuntimeError: Cannot access "https://github.com/mayureshnadkar-beep/spring-ai-ollama": Not Found

---
## 🔍 6. Vulnerability Detection Engine

In [16]:
# ═══════════════════════════════════════════════════════════
#  PATTERN DEFINITIONS
# ═══════════════════════════════════════════════════════════

# Each entry: (display_name, regex, severity, vuln_type, reference_url)

SECRET_PATTERNS = [
    (
        'Hardcoded Password',
        r'(?i)(password|passwd|pwd)\s*[=:]\s*[\'"]([^\s\'"]{4,})[\'"]',
        'Critical',
        'Hardcoded Secret',
        'https://cwe.mitre.org/data/definitions/259.html'
    ),
    (
        'Hardcoded API Key',
        r'(?i)(apikey|api[_\-]?key|api[_\-]?secret)\s*[=:]\s*[\'"]([A-Za-z0-9\-_]{16,80})[\'"]',
        'Critical',
        'Hardcoded Secret',
        'https://owasp.org/www-community/vulnerabilities/Use_of_hard-coded_credentials'
    ),
    (
        'Hardcoded Token',
        r'(?i)(token|auth[_\-]?token|access[_\-]?token)\s*[=:]\s*[\'"]([A-Za-z0-9\-_.]{16,})[\'"]',
        'Critical',
        'Hardcoded Secret',
        'https://cwe.mitre.org/data/definitions/798.html'
    ),
    (
        'Hardcoded Secret Value',
        r'(?i)(secret|client[_\-]?secret)\s*[=:]\s*[\'"]([A-Za-z0-9\-_]{8,})[\'"]',
        'High',
        'Hardcoded Secret',
        'https://cwe.mitre.org/data/definitions/798.html'
    ),
    (
        'AWS Access Key ID',
        r'(?<![A-Z0-9])AKIA[0-9A-Z]{16}(?![A-Z0-9])',
        'Critical',
        'Cloud Credential Exposure',
        'https://docs.aws.amazon.com/general/latest/gr/aws-security-credentials.html'
    ),
    (
        'AWS Secret Access Key',
        r'(?i)aws.{0,20}secret.{0,20}[=:]\s*[\'"]?[A-Za-z0-9/+=]{40}',
        'Critical',
        'Cloud Credential Exposure',
        'https://docs.aws.amazon.com/general/latest/gr/aws-security-credentials.html'
    ),
    (
        'Private Key Embedded',
        r'-----BEGIN (RSA |EC )?PRIVATE KEY-----',
        'Critical',
        'Cryptographic Key Exposure',
        'https://cwe.mitre.org/data/definitions/321.html'
    ),
    (
        'Database URL with Credentials',
        r'(?i)(jdbc:|mongodb://|mysql://|postgresql://).{0,40}:[^\s@]{4,}@',
        'Critical',
        'Hardcoded Secret',
        'https://owasp.org/www-community/vulnerabilities/Use_of_hard-coded_credentials'
    ),
]

JAVA_PATTERNS = [
    (
        'OS Command Injection (exec)',
        r'Runtime\.getRuntime\(\)\.exec\s*\(',
        'Critical',
        'Command Injection',
        'https://cwe.mitre.org/data/definitions/78.html'
    ),
    (
        'SQL Injection Risk (String concat)',
        r'(?i)(createQuery|createNativeQuery|executeQuery|prepareStatement)\s*\([^)]*\+',
        'Critical',
        'SQL Injection',
        'https://owasp.org/www-community/attacks/SQL_Injection'
    ),
    (
        'SQL Injection via Statement',
        r'Statement\s+\w+\s*=.*;\s*\w+\.execute[^(]*\(.*\+',
        'Critical',
        'SQL Injection',
        'https://owasp.org/www-community/attacks/SQL_Injection'
    ),
    (
        'Weak Hash — MD5',
        r'MessageDigest\.getInstance\s*\(\s*[\'"]MD5[\'"]',
        'High',
        'Weak Cryptography',
        'https://cwe.mitre.org/data/definitions/327.html'
    ),
    (
        'Weak Hash — SHA-1',
        r'MessageDigest\.getInstance\s*\(\s*[\'"]SHA-?1[\'"]',
        'High',
        'Weak Cryptography',
        'https://cwe.mitre.org/data/definitions/327.html'
    ),
    (
        'Stack Trace Exposure (printStackTrace)',
        r'\.printStackTrace\s*\(',
        'Medium',
        'Information Disclosure',
        'https://cwe.mitre.org/data/definitions/209.html'
    ),
    (
        'Open CORS Policy (@CrossOrigin *)',
        r'@CrossOrigin\s*\([^)]*[\*"]\*["\)]',
        'High',
        'Misconfiguration',
        'https://owasp.org/www-project-web-security-testing-guide/v42/4-Web_Application_Security_Testing/11-Client_Side_Testing/07-Testing_Cross_Origin_Resource_Sharing'
    ),
    (
        'Open CORS (origins = "*")',
        r'@CrossOrigin',
        'Medium',
        'Misconfiguration',
        'https://cwe.mitre.org/data/definitions/942.html'
    ),
    (
        'Debug Mode Enabled',
        r'(?i)(debug\s*=\s*true|logging\.level.*DEBUG)',
        'Medium',
        'Misconfiguration',
        'https://cwe.mitre.org/data/definitions/489.html'
    ),
    (
        'Insecure Random (java.util.Random)',
        r'new\s+Random\s*\(',
        'Medium',
        'Weak Randomness',
        'https://cwe.mitre.org/data/definitions/330.html'
    ),
    (
        'Object Deserialization Risk',
        r'ObjectInputStream\s*\(',
        'High',
        'Insecure Deserialization',
        'https://owasp.org/www-community/vulnerabilities/Deserialization_of_untrusted_data'
    ),
    (
        'XML External Entity (XXE) Risk',
        r'(?i)(DocumentBuilderFactory|SAXParserFactory|XMLInputFactory)\.newInstance',
        'High',
        'XXE Injection',
        'https://owasp.org/www-community/vulnerabilities/XML_External_Entity_(XXE)_Processing'
    ),
    (
        'SSL Verification Disabled',
        r'(?i)(setHostnameVerifier|trustAllCertificates|ALLOW_ALL_HOSTNAME_VERIFIER)',
        'High',
        'Insecure Transport',
        'https://cwe.mitre.org/data/definitions/297.html'
    ),
    (
        'Actuator Endpoints Exposed',
        r'(?i)management\.endpoints\.web\.exposure\.include\s*=\s*[\*]',
        'High',
        'Misconfiguration',
        'https://docs.spring.io/spring-boot/docs/current/reference/html/actuator.html'
    ),
    (
        'Spring Security Disabled (permitAll)',
        r'permitAll\s*\(\s*\)',
        'High',
        'Broken Access Control',
        'https://owasp.org/www-project-top-ten/2017/A5_2017-Broken_Access_Control'
    ),
    (
        'Plaintext Credentials in Properties',
        r'(?i)spring\.(datasource|security)\.password\s*=\s*\S+',
        'High',
        'Hardcoded Secret',
        'https://cwe.mitre.org/data/definitions/256.html'
    ),
]

ALL_PATTERNS = SECRET_PATTERNS + JAVA_PATTERNS

print(f'✅ Vulnerability patterns loaded.')
print(f'   Secret patterns : {len(SECRET_PATTERNS)}')
print(f'   Java patterns   : {len(JAVA_PATTERNS)}')
print(f'   Total patterns  : {len(ALL_PATTERNS)}')

✅ Vulnerability patterns loaded.
   Secret patterns : 8
   Java patterns   : 16
   Total patterns  : 24


In [17]:
def scan_source_file(file_item, repo_name: str) -> list:
    """
    Apply all regex patterns to a single source file.
    Returns a list of raw vulnerability dicts (AI fields blank).
    """
    content = decode_content(file_item)
    if not content:
        return []

    lines  = content.splitlines()
    found  = []
    seen   = set()   # avoid duplicate pattern hits in same file

    for name, pattern, severity, vuln_type, ref in ALL_PATTERNS:
        regex = re.compile(pattern, re.IGNORECASE)
        for line_no, line in enumerate(lines, start=1):
            stripped = line.strip()
            # Skip pure comment lines
            if stripped.startswith(('//', '#', '*', '<!--')):
                continue
            if regex.search(line):
                key = (name, file_item.path)
                if key in seen:
                    break
                seen.add(key)

                # Build safe snippet (redact secrets)
                snippet = line[:150].strip()
                if vuln_type in ('Hardcoded Secret', 'Cloud Credential Exposure',
                                  'Cryptographic Key Exposure'):
                    snippet = re.sub(
                        r'([=:\s][\'"]?)([A-Za-z0-9+/=\-_]{6,})',
                        r'\1[REDACTED]', snippet
                    )

                found.append({
                    'Repository Name'   : repo_name,
                    'File Name'         : file_item.path,
                    'Line Number'       : line_no,
                    'Vulnerability Type': vuln_type,
                    'Vulnerability Name': name,
                    'Severity'          : severity,
                    'Risk Score'        : SEVERITY_SCORE[severity],
                    'Code Snippet'      : snippet,
                    'Reference Link'    : ref,
                    'AI Explanation'    : '',
                    'Risk Impact'       : '',
                    'Recommended Fix'   : '',
                    'Secure Code Example': '',
                })
                break   # one hit per pattern per file
    return found


print('✅ scan_source_file() ready.')

✅ scan_source_file() ready.


In [ ]:
# ── Dependency scanner — pom.xml → OSV API ───────────────────────────

def parse_pom_dependencies(content: str) -> list:
    """Extract (groupId, artifactId, version) tuples from pom.xml."""
    deps = []
    try:
        root = ET.fromstring(content)
        ns   = {'m': 'http://maven.apache.org/POM/4.0.0'}

        def find_text(el, tag):
            child = el.find(f'm:{tag}', ns) or el.find(tag)
            return child.text.strip() if child is not None and child.text else ''

        dep_section = root.find('.//m:dependencies', ns) or root.find('.//dependencies')
        if dep_section is None:
            return deps
        for dep in dep_section:
            group   = find_text(dep, 'groupId')
            artifact= find_text(dep, 'artifactId')
            version = find_text(dep, 'version')
            if group and artifact:
                # Clean ${...} placeholders
                version = '' if version.startswith('$') else version
                deps.append({'group': group, 'artifact': artifact, 'version': version})
    except ET.ParseError as exc:
        print(f'   ⚠️  pom.xml parse error: {exc}')
    return deps


def osv_query(package_name: str, version: str, ecosystem: str = 'Maven') -> list:
    """Query OSV API for known CVEs. Returns list of vuln dicts."""
    payload = {'package': {'name': package_name, 'ecosystem': ecosystem}}
    if version:
        payload['version'] = version
    try:
        resp = requests.post(OSV_API, json=payload, timeout=10)
        if resp.status_code == 200:
            return resp.json().get('vulns', [])
    except requests.RequestException:
        pass
    return []


def scan_pom_file(file_item, repo_name: str) -> list:
    """Scan pom.xml: parse deps, query OSV, return vulnerability list."""
    content = decode_content(file_item)
    deps    = parse_pom_dependencies(content)
    vulns   = []

    for dep in tqdm(deps, desc=f'OSV: {file_item.name}', leave=False):
        pkg_name = f"{dep['group']}:{dep['artifact']}"
        hits     = osv_query(pkg_name, dep['version'], 'Maven')
        for hit in hits[:2]:
            # Determine severity from CVSS
            cvss   = 0.0
            cve_id = next((a for a in hit.get('aliases', []) if a.startswith('CVE-')), hit.get('id', ''))
            for sev in hit.get('severity', []):
                nums = re.findall(r'[0-9]+\.?[0-9]*', sev.get('score', ''))
                if nums:
                    cvss = float(nums[0])
            severity = ('Critical' if cvss >= 9 else
                        'High'     if cvss >= 7 else
                        'Medium'   if cvss >= 4 else 'Low')

            # Fixed version
            fix_ver = ''
            for affected in hit.get('affected', []):
                for rng in affected.get('ranges', []):
                    for ev in rng.get('events', []):
                        if 'fixed' in ev:
                            fix_ver = ev['fixed']

            summary = hit.get('summary', 'Known vulnerability in this dependency.')

            vulns.append({
                'Repository Name'   : repo_name,
                'File Name'         : file_item.path,
                'Line Number'       : 'N/A',
                'Vulnerability Type': 'Dependency Vulnerability',
                'Vulnerability Name': f'{pkg_name} — {cve_id}',
                'Severity'          : severity,
                'Risk Score'        : SEVERITY_SCORE[severity],
                'Code Snippet'      : f'<dependency>{pkg_name}:{dep["version"]}</dependency>',
                'Reference Link'    : f'https://osv.dev/vulnerability/{hit.get("id","")}',
                'AI Explanation'    : '',
                'Risk Impact'       : summary,
                'Recommended Fix'   : f'Upgrade to {fix_ver}' if fix_ver else 'Check latest secure version',
                'Secure Code Example': f'<version>{fix_ver}</version>' if fix_ver else '# Upgrade dependency',
            })
        time.sleep(0.15)   # respect OSV rate limits
    return vulns


print('✅ Dependency scanner (OSV API) ready.')

In [18]:
# ── RUN DETECTION — Phase 1 & 2 ─────────────────────────────────────

raw_vulns = []

print('\n🔍 Phase 1 — Scanning source code files...')
for fi in tqdm(repo_data['code_files'], desc='Code scan', unit='file'):
    raw_vulns.extend(scan_source_file(fi, TARGET_REPO))

print(f'   Found {len(raw_vulns)} code-level findings.')

print('\n📦 Phase 2 — Scanning dependency files (pom.xml → OSV API)...')
dep_count_before = len(raw_vulns)
for fi in repo_data['dep_files']:
    if fi.name.lower() == 'pom.xml':
        raw_vulns.extend(scan_pom_file(fi, TARGET_REPO))

dep_vulns = len(raw_vulns) - dep_count_before
print(f'   Found {dep_vulns} dependency CVEs.')
print(f'\n   ✅ Total raw findings : {len(raw_vulns)}')


🔍 Phase 1 — Scanning source code files...


Code scan:   0%|          | 0/4 [00:00<?, ?file/s]

   Found 0 code-level findings.

📦 Phase 2 — Scanning dependency files (pom.xml → OSV API)...


NameError: name 'scan_pom_file' is not defined

---
## 🤖 7. Hugging Face AI Analysis

For each unique vulnerability pattern we query `google/flan-t5-large` via the Hugging Face Inference API.  
The model returns a structured explanation, severity rating, risk impact, remediation steps and a secure code example.

If no HF token is provided, a high-quality **rule-based fallback** fills these fields automatically.

In [ ]:
HF_ENDPOINT = f'{HF_API_BASE}/{HF_MODEL}'

# Cache to avoid paying the same AI call twice
_ai_cache: dict = {}


PROMPT_TEMPLATE = """\
You are a senior application security engineer. Analyze the following security vulnerability found in a Java Spring Boot project.

Vulnerability: {name}
Type: {vtype}
File: {fname}
Code snippet: {snippet}
Detected severity: {severity}

Provide a concise analysis with these five parts:
1. EXPLANATION: What this vulnerability is and why it exists.
2. SEVERITY: Confirm or revise severity (Low/Medium/High/Critical).
3. RISK IMPACT: What an attacker could do if this is exploited.
4. REMEDIATION: 2-3 concrete steps to fix it.
5. SECURE CODE: A short Java/properties code example showing the safe implementation.
"""


# ── Rule-based fallback library ─────────────────────────────────────
FALLBACK_DB = {
    'Hardcoded Password': (
        'A password is stored as plaintext directly in source code, making it visible to anyone with code access.',
        'Critical',
        'Attackers who read the source (via repo breach or code leak) gain immediate database/API access.',
        '1. Move credentials to environment variables or a secret manager (e.g. Vault, AWS Secrets Manager).\n2. Use Spring @Value with ${ENV_VAR}.\n3. Never commit .env files to version control.',
        'spring.datasource.password=${DB_PASSWORD}  # loaded from env'
    ),
    'OS Command Injection (exec)': (
        'Runtime.exec() runs OS commands and is dangerous when any part of the command includes user-supplied input.',
        'Critical',
        'Remote code execution — attacker can run arbitrary OS commands, exfiltrate data or destroy the system.',
        '1. Avoid Runtime.exec() entirely; use a Java library instead.\n2. If unavoidable, use a whitelist of allowed commands.\n3. Never concatenate user input into the command string.',
        'ProcessBuilder pb = new ProcessBuilder(List.of("/usr/bin/safe-tool", sanitisedArg));\npb.start();'
    ),
    'SQL Injection Risk (String concat)': (
        'Building SQL queries using string concatenation with user-controlled data allows attackers to inject SQL.',
        'Critical',
        'Full database dump, authentication bypass, data deletion or privilege escalation.',
        '1. Use parameterised queries or PreparedStatement exclusively.\n2. Use JPA/Hibernate named parameters.\n3. Apply input validation.',
        'String q = "SELECT * FROM users WHERE id = ?";\nPreparedStatement ps = conn.prepareStatement(q);\nps.setInt(1, userId);'
    ),
    'Weak Hash — MD5': (
        'MD5 is a cryptographically broken hash function — collisions can be found in seconds on commodity hardware.',
        'High',
        'Password hashes can be reversed; integrity checks can be forged.',
        '1. Replace MD5 with SHA-256 or SHA-3.\n2. For passwords, use BCrypt, SCrypt, or Argon2.\n3. Add a unique per-user salt.',
        'MessageDigest md = MessageDigest.getInstance("SHA-256");\nbyte[] hash = md.digest(data);'
    ),
    'Stack Trace Exposure (printStackTrace)': (
        'Printing stack traces to standard output leaks internal class names, file paths and library versions.',
        'Medium',
        'Attackers gain reconnaissance information to craft targeted exploits.',
        '1. Replace with a structured logger (SLF4J + Logback).\n2. Return generic error messages to clients.\n3. Centralise exception handling with @ControllerAdvice.',
        'log.error("Operation failed: {}", e.getMessage(), e);  // use logger, not e.printStackTrace()'
    ),
    'Open CORS Policy (@CrossOrigin *)': (
        '@CrossOrigin("*") allows any web origin to make credentialed cross-site requests to this endpoint.',
        'High',
        'CSRF attacks and cross-origin data theft from authenticated sessions.',
        '1. Restrict origins to known trusted domains.\n2. Configure CORS globally via WebMvcConfigurer.\n3. Never use wildcard in production.',
        '@CrossOrigin(origins = "https://app.example.com")\n@RestController\npublic class MyController { ... }'
    ),
    'Actuator Endpoints Exposed': (
        'Exposing all Spring Actuator endpoints (management.endpoints.web.exposure.include=*) in production is dangerous.',
        'High',
        'Attackers can read environment variables, heap dumps, and trigger application shutdown.',
        '1. Only expose necessary endpoints (health, info).\n2. Secure actuator with Spring Security.\n3. Move actuator to a separate management port not reachable from the internet.',
        'management.endpoints.web.exposure.include=health,info\nmanagement.endpoint.health.show-details=when_authorized'
    ),
    'Object Deserialization Risk': (
        'Java ObjectInputStream deserializes arbitrary class graphs, which attackers can exploit via gadget chains.',
        'High',
        'Remote code execution if attacker controls the serialized byte stream.',
        '1. Avoid Java native serialization; use JSON/Protobuf instead.\n2. If unavoidable, implement a ValidatingObjectInputStream whitelist.\n3. Apply serial-killer or other deserialization filters.',
        '// Prefer Jackson or Gson:\nMyObj obj = objectMapper.readValue(jsonBytes, MyObj.class);'
    ),
    'Debug Mode Enabled': (
        'Debug logging or debug mode left enabled in production causes verbose output and disables optimisations.',
        'Medium',
        'Sensitive data (SQL queries, tokens, stack traces) may appear in logs accessible to attackers.',
        '1. Set logging.level.root=WARN for production.\n2. Use Spring Profiles to separate dev vs prod configs.\n3. Audit log configuration before each release.',
        '# application-prod.properties\nlogging.level.root=WARN\nspring.jpa.show-sql=false'
    ),
    'Spring Security Disabled (permitAll)': (
        'permitAll() removes all authentication requirements from the matched URL pattern.',
        'High',
        'Unauthenticated users gain access to sensitive endpoints.',
        '1. Audit every permitAll() call and apply the least-privilege principle.\n2. Restrict admin/internal paths to specific roles.\n3. Enable CSRF protection for state-changing operations.',
        '.requestMatchers("/public/**").permitAll()\n.requestMatchers("/admin/**").hasRole("ADMIN")\n.anyRequest().authenticated()'
    ),
}


def _fallback_analysis(vuln_name: str, severity: str) -> tuple:
    """Return rule-based analysis when AI is unavailable."""
    # Try exact match, then partial match
    entry = FALLBACK_DB.get(vuln_name)
    if not entry:
        for key, val in FALLBACK_DB.items():
            if any(w in vuln_name for w in key.split()):
                entry = val
                break
    if entry:
        explanation, sev, impact, fix, code = entry
    else:
        explanation = f'{vuln_name} is a security vulnerability that requires immediate attention.'
        sev         = severity
        impact      = 'Could allow unauthorised access or data leakage.'
        fix         = '1. Review the flagged code.\n2. Follow OWASP guidelines.\n3. Add security unit tests.'
        code        = '// Refactor using security best practices'
    return explanation, sev, impact, fix, code


def get_ai_analysis(vuln: dict) -> tuple:
    """
    Query HF Inference API (flan-t5-large) for a structured security analysis.
    Returns (explanation, severity, risk_impact, remediation, secure_code).
    Falls back to rule-based answers on any error.
    """
    cache_key = f"{vuln['Vulnerability Name']}|{vuln['Vulnerability Type']}"
    if cache_key in _ai_cache:
        return _ai_cache[cache_key]

    # No token → use fallback immediately
    if not hf_headers:
        result = _fallback_analysis(vuln['Vulnerability Name'], vuln['Severity'])
        _ai_cache[cache_key] = result
        return result

    prompt = PROMPT_TEMPLATE.format(
        name    = vuln['Vulnerability Name'],
        vtype   = vuln['Vulnerability Type'],
        fname   = os.path.basename(vuln['File Name']),
        snippet = vuln['Code Snippet'][:250],
        severity= vuln['Severity'],
    )

    payload = {
        'inputs'    : prompt,
        'parameters': {
            'max_new_tokens': HF_MAX_TOKENS,
            'temperature'   : 0.3,
            'do_sample'     : False,
        }
    }

    try:
        time.sleep(HF_CALL_DELAY)
        resp = requests.post(HF_ENDPOINT, headers=hf_headers,
                             json=payload, timeout=60)

        if resp.status_code == 503:
            # Model is loading — wait and retry once
            wait_time = resp.json().get('estimated_time', 20)
            print(f'   ⏳ Model loading, waiting {wait_time:.0f}s...')
            time.sleep(min(wait_time, 30))
            resp = requests.post(HF_ENDPOINT, headers=hf_headers,
                                  json=payload, timeout=60)

        if resp.status_code != 200:
            raise RuntimeError(f'HF API returned {resp.status_code}: {resp.text[:200]}')

        raw = resp.json()
        if isinstance(raw, list):
            text = raw[0].get('generated_text', '')
        elif isinstance(raw, dict):
            text = raw.get('generated_text', '')
        else:
            text = str(raw)

        # ── Parse structured sections from model output ──────────────
        def extract(label: str, next_labels: list, full_text: str) -> str:
            pattern = rf'(?i){re.escape(label)}[:\.]?\s*(.+?)(?=(?:{"|".join(re.escape(l) for l in next_labels)})[:\.]|$)'
            m = re.search(pattern, full_text, re.DOTALL)
            return m.group(1).strip()[:600] if m else ''

        all_labels = ['EXPLANATION', 'SEVERITY', 'RISK IMPACT', 'REMEDIATION', 'SECURE CODE']
        explanation = extract('EXPLANATION', all_labels[1:], text)
        sev_raw     = extract('SEVERITY',    all_labels[2:], text)
        impact      = extract('RISK IMPACT', all_labels[3:], text)
        remediation = extract('REMEDIATION', all_labels[4:], text)
        secure_code = extract('SECURE CODE', [],             text)

        # Normalise severity
        sev_clean = vuln['Severity']
        for s in ['Critical', 'High', 'Medium', 'Low']:
            if s.lower() in sev_raw.lower():
                sev_clean = s
                break

        # Fall back to rule-based if model returned mostly empty
        if not explanation or len(explanation) < 20:
            fb = _fallback_analysis(vuln['Vulnerability Name'], vuln['Severity'])
            explanation = explanation or fb[0]
            sev_clean   = sev_clean   or fb[1]
            impact      = impact      or fb[2]
            remediation = remediation or fb[3]
            secure_code = secure_code or fb[4]

        result = (explanation, sev_clean, impact, remediation, secure_code or '# See remediation steps')
        _ai_cache[cache_key] = result
        return result

    except Exception as exc:
        print(f'   ⚠️  HF API error ({type(exc).__name__}): {str(exc)[:100]}  → using fallback')
        result = _fallback_analysis(vuln['Vulnerability Name'], vuln['Severity'])
        _ai_cache[cache_key] = result
        return result


print('✅ Hugging Face AI analysis module ready.')
print(f'   Model    : {HF_MODEL}')
print(f'   Endpoint : {HF_ENDPOINT}')
print(f'   Token    : {"Configured" if hf_headers else "Not provided — fallback mode"}')

In [ ]:
# ── RUN AI ANALYSIS — Phase 3 ────────────────────────────────────────

print(f'\n🤖 Phase 3 — Running AI analysis on {len(raw_vulns)} finding(s)...')
print(f'   (Unique cache entries shared across duplicate patterns)\n')

for vuln in tqdm(raw_vulns, desc='AI Analysis', unit='vuln'):
    explanation, severity, impact, fix, code = get_ai_analysis(vuln)
    vuln['AI Explanation']     = explanation
    vuln['Severity']           = severity          # AI may revise
    vuln['Risk Score']         = SEVERITY_SCORE.get(severity, vuln['Risk Score'])
    vuln['Risk Impact']        = impact
    vuln['Recommended Fix']    = fix
    vuln['Secure Code Example']= code

print(f'\n✅ AI analysis complete.  Cache hits saved: {len(_ai_cache)} unique prompts.')

---
## 📊 8. Build DataFrame & Risk Scoring

In [ ]:
FINAL_COLUMNS = [
    'Repository Name', 'File Name', 'Line Number',
    'Vulnerability Type', 'Vulnerability Name', 'Severity', 'Risk Score',
    'Code Snippet', 'AI Explanation', 'Risk Impact',
    'Recommended Fix', 'Secure Code Example', 'Reference Link'
]

if raw_vulns:
    df = pd.DataFrame(raw_vulns, columns=FINAL_COLUMNS)
    # Sort: Critical → High → Medium → Low
    sev_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
    df['_ord'] = df['Severity'].map(sev_order).fillna(4)
    df = df.sort_values('_ord').drop(columns='_ord').reset_index(drop=True)
else:
    df = pd.DataFrame(columns=FINAL_COLUMNS)


# ── Risk score engine ────────────────────────────────────────────────
total_score = int(df['Risk Score'].sum()) if not df.empty else 0
sev_counts  = df['Severity'].value_counts().to_dict() if not df.empty else {}
type_counts = df['Vulnerability Type'].value_counts().to_dict() if not df.empty else {}

classification, cls_color = 'Safe', '#2ECC71'
for threshold, label, color in RISK_BANDS:
    if total_score <= threshold:
        classification, cls_color = label, color
        break

risk_report = {
    'total_score'   : total_score,
    'classification': classification,
    'cls_color'     : cls_color,
    'sev_counts'    : sev_counts,
    'type_counts'   : type_counts,
    'top_files'     : df['File Name'].value_counts().head(5).to_dict() if not df.empty else {},
    'meta'          : repo_data['repo_meta'],
}

print('\n' + '─'*56)
print('  📊  RISK SCORING SUMMARY')
print('─'*56)
print(f'  Repository       : {repo_data["repo_meta"]["full_name"]}')
print(f'  Total Risk Score : {total_score}')
print(f'  Classification   : {classification}')
print(f'  Total Findings   : {len(df)}')
print('─'*56)
for sev in ['Critical', 'High', 'Medium', 'Low']:
    n   = sev_counts.get(sev, 0)
    bar = '█' * min(n, 35)
    print(f'  {sev:<10}: {bar} {n}')
print('─'*56)
print(df[['File Name', 'Vulnerability Name', 'Severity', 'Risk Score']].to_string(index=False))

---
## 📈 9. Visualization

In [ ]:
def draw_visualizations(df: pd.DataFrame, risk_report: dict):
    """Render the full 6-panel security dashboard."""

    if df.empty:
        print('ℹ️  No findings — skipping charts.')
        return

    SEV_ORDER  = ['Critical', 'High', 'Medium', 'Low']
    sc         = risk_report['sev_counts']
    tc         = risk_report['type_counts']
    sev_present= [s for s in SEV_ORDER if s in sc]
    sev_vals   = [sc[s] for s in sev_present]
    bar_colors = [SEVERITY_COLOR[s] for s in sev_present]

    fig = plt.figure(figsize=(22, 16), facecolor='#0F172A')
    fig.suptitle(
        f'🔐  Security Scan Dashboard — {risk_report["meta"]["full_name"]}',
        fontsize=17, fontweight='bold', color='white', y=0.98
    )

    AXBG = '#1E293B'

    def style_ax(ax, title):
        ax.set_facecolor(AXBG)
        ax.set_title(title, color='white', fontweight='bold', pad=10, fontsize=11)
        ax.tick_params(colors='#94A3B8')
        for sp in ax.spines.values():
            sp.set_color('#334155')

    # ── 1. Severity bar chart ────────────────────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    style_ax(ax1, 'Vulnerabilities by Severity')
    bars = ax1.bar(sev_present, sev_vals, color=bar_colors,
                   edgecolor='white', linewidth=0.6, width=0.55)
    ax1.set_ylabel('Count', color='#94A3B8')
    for b, v in zip(bars, sev_vals):
        ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
                 str(v), ha='center', color='white', fontweight='bold', fontsize=12)

    # ── 2. Severity pie chart ────────────────────────────────────────
    ax2 = fig.add_subplot(2, 3, 2)
    style_ax(ax2, 'Severity Distribution')
    wedges, texts, autos = ax2.pie(
        sev_vals, labels=sev_present, colors=bar_colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': '#0F172A', 'linewidth': 2}
    )
    for t in texts + autos:
        t.set_color('white')
        t.set_fontsize(10)

    # ── 3. Vulnerability types horizontal bar ────────────────────────
    ax3 = fig.add_subplot(2, 3, 3)
    style_ax(ax3, 'Vulnerability Categories')
    types  = list(tc.keys())[:8]
    tvals  = [tc[t] for t in types]
    tclrs  = plt.cm.Set2(np.linspace(0, 1, len(types)))
    hb     = ax3.barh([t[:28] for t in types], tvals, color=tclrs,
                       edgecolor='white', linewidth=0.4)
    ax3.set_xlabel('Count', color='#94A3B8')
    for b, v in zip(hb, tvals):
        ax3.text(b.get_width() + 0.05, b.get_y() + b.get_height()/2,
                 str(v), va='center', color='white', fontsize=9)

    # ── 4. Top vulnerable files ──────────────────────────────────────
    ax4 = fig.add_subplot(2, 3, 4)
    style_ax(ax4, 'Top 5 Vulnerable Files')
    tf = risk_report['top_files']
    if tf:
        fnames = [os.path.basename(k)[:30] for k in tf]
        fvals  = list(tf.values())
        ax4.barh(fnames, fvals, color='#38BDF8', edgecolor='white', linewidth=0.4)
        ax4.set_xlabel('Findings', color='#94A3B8')

    # ── 5. Risk gauge ────────────────────────────────────────────────
    ax5 = fig.add_subplot(2, 3, 5)
    style_ax(ax5, 'Repository Risk Status')
    ax5.set_xlim(0, 1); ax5.set_ylim(0, 1); ax5.axis('off')
    score = risk_report['total_score']
    cls   = risk_report['classification']
    cc    = risk_report['cls_color']
    circle_bg   = plt.Circle((0.5, 0.46), 0.32, color='#0F172A', fill=True)
    circle_ring = plt.Circle((0.5, 0.46), 0.32, color=cc, linewidth=8, fill=False)
    ax5.add_patch(circle_bg); ax5.add_patch(circle_ring)
    ax5.text(0.5, 0.53, str(score), ha='center', va='center',
             fontsize=38, fontweight='bold', color=cc)
    ax5.text(0.5, 0.38, 'RISK SCORE', ha='center',
             fontsize=9, color='#94A3B8')
    ax5.text(0.5, 0.11, cls, ha='center', fontsize=13,
             fontweight='bold', color=cc,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#1E293B',
                       edgecolor=cc, linewidth=2))

    # ── 6. Stacked bar — severity by type ────────────────────────────
    ax6 = fig.add_subplot(2, 3, 6)
    style_ax(ax6, 'Severity by Category (Stacked)')
    pivot = df.groupby(['Vulnerability Type', 'Severity']).size().unstack(fill_value=0)
    for s in SEV_ORDER:
        if s not in pivot.columns:
            pivot[s] = 0
    pivot = pivot[[s for s in SEV_ORDER if s in pivot.columns]]
    bottom = np.zeros(len(pivot))
    xlabels = [t[:22] for t in pivot.index]
    for sev in SEV_ORDER:
        if sev in pivot.columns:
            vals = pivot[sev].values
            ax6.bar(xlabels, vals, bottom=bottom, label=sev,
                    color=SEVERITY_COLOR[sev], edgecolor='white', linewidth=0.3)
            bottom += vals
    ax6.tick_params(axis='x', colors='#94A3B8', rotation=18, labelsize=8)
    ax6.tick_params(axis='y', colors='#94A3B8')
    ax6.legend(loc='upper right', framealpha=0.4, labelcolor='white',
               facecolor='#1E293B', edgecolor='#334155', fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('security_dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0F172A')
    plt.show()
    print('\n📊 Dashboard saved → security_dashboard.png')


draw_visualizations(df, risk_report)

---
## 📄 10. Excel Report Export

In [ ]:
def auto_width(ws, min_w=10, max_w=65):
    """Auto-fit all column widths based on cell content."""
    for col_cells in ws.columns:
        best = min_w
        for cell in col_cells:
            if cell.value:
                best = max(best, min(len(str(cell.value).split('\n')[0]) + 2, max_w))
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = best


def build_excel_report(df: pd.DataFrame, risk_report: dict, filepath: str):
    """
    Generate a professional 2-sheet Excel workbook:
      Sheet 1 — Executive Summary
      Sheet 2 — Detailed Findings
    """
    wb   = Workbook()
    meta = risk_report['meta']
    sc   = risk_report['sev_counts']
    thin = Side(style='thin', color='D1D5DB')
    bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

    def hdr_fill(hex_col):
        return PatternFill('solid', fgColor=hex_col)

    def cell_font(bold=False, size=10, color='000000', name='Arial'):
        return Font(bold=bold, size=size, color=color, name=name)

    def write_kv(ws, row, label, value, label_bg='F1F5F9',
                 val_bg='FFFFFF', bold_val=False, val_color='000000'):
        lc = ws.cell(row=row, column=1, value=label)
        lc.font   = cell_font(bold=True, size=10)
        lc.fill   = hdr_fill(label_bg)
        lc.border = bdr
        lc.alignment = Alignment(vertical='center', indent=1)
        ws.merge_cells(start_row=row, start_column=2,
                       end_row=row, end_column=8)
        vc = ws.cell(row=row, column=2, value=value)
        vc.font   = cell_font(bold=bold_val, size=10, color=val_color)
        vc.fill   = hdr_fill(val_bg)
        vc.border = bdr
        vc.alignment = Alignment(vertical='center', wrap_text=False)

    # ════════════════════════════════════════════════════════════
    # SHEET 1 — EXECUTIVE SUMMARY
    # ════════════════════════════════════════════════════════════
    ws1 = wb.active
    ws1.title = '📊 Executive Summary'
    ws1.sheet_view.showGridLines = False

    # Banner
    ws1.merge_cells('A1:H1')
    c = ws1['A1']
    c.value     = '🔐  AI-POWERED GITHUB SECURITY VULNERABILITY REPORT'
    c.font      = cell_font(bold=True, size=16, color='FFFFFF')
    c.fill      = hdr_fill('0F172A')
    c.alignment = Alignment(horizontal='center', vertical='center')
    ws1.row_dimensions[1].height = 42

    ws1.merge_cells('A2:H2')
    c = ws1['A2']
    c.value     = (f'Generated: {meta["scanned_at"]}  |  '
                   f'AI Model: {HF_MODEL}  |  OSV Dependency Scanning')
    c.font      = cell_font(size=9, color='94A3B8')
    c.fill      = hdr_fill('1E293B')
    c.alignment = Alignment(horizontal='center')
    ws1.row_dimensions[2].height = 20

    # Section: Repo Info
    ws1.merge_cells('A4:H4')
    c = ws1['A4']
    c.value     = '📌  REPOSITORY INFORMATION'
    c.font      = cell_font(bold=True, size=11, color='FFFFFF')
    c.fill      = hdr_fill('1E3A5F')
    c.alignment = Alignment(indent=1, vertical='center')
    ws1.row_dimensions[4].height = 24

    repo_rows = [
        ('Repository',  meta['full_name']),
        ('URL',         meta['url']),
        ('Language',    meta['language']),
        ('Description', meta['description']),
        ('Stars',       str(meta['stars'])),
        ('Forks',       str(meta['forks'])),
        ('Repo Size',   f"{meta['size_kb']:,} KB"),
    ]
    for i, (lbl, val) in enumerate(repo_rows, start=5):
        write_kv(ws1, i, lbl, val)
        ws1.row_dimensions[i].height = 18

    # Section: Risk Assessment
    r = 13
    ws1.merge_cells(f'A{r}:H{r}')
    c = ws1[f'A{r}']
    c.value     = '⚠️  RISK ASSESSMENT'
    c.font      = cell_font(bold=True, size=11, color='FFFFFF')
    c.fill      = hdr_fill('3B1F4E')
    c.alignment = Alignment(indent=1, vertical='center')
    ws1.row_dimensions[r].height = 24

    # Total score
    write_kv(ws1, r+1, 'Total Risk Score', str(risk_report['total_score']),
             bold_val=True)
    # Classification — coloured
    cls_hex = risk_report['cls_color'].replace('#','')
    write_kv(ws1, r+2, 'Risk Classification', risk_report['classification'],
             val_bg=cls_hex, val_color='FFFFFF', bold_val=True)
    write_kv(ws1, r+3, 'Total Vulnerabilities', str(len(df)), bold_val=True)

    for idx, sev in enumerate(['Critical', 'High', 'Medium', 'Low'], start=r+4):
        cnt = sc.get(sev, 0)
        write_kv(ws1, idx, f'  {sev}', str(cnt),
                 val_bg=SEVERITY_BG_HEX[sev],
                 val_color=SEVERITY_FG_HEX[sev],
                 bold_val=cnt > 0)
        ws1.row_dimensions[idx].height = 18

    # Section: Finding Breakdown by Type
    r2 = r + 9
    ws1.merge_cells(f'A{r2}:H{r2}')
    c = ws1[f'A{r2}']
    c.value     = '📁  BREAKDOWN BY VULNERABILITY CATEGORY'
    c.font      = cell_font(bold=True, size=11, color='FFFFFF')
    c.fill      = hdr_fill('1A3A2A')
    c.alignment = Alignment(indent=1, vertical='center')
    ws1.row_dimensions[r2].height = 24

    for idx, (vtype, cnt) in enumerate(risk_report['type_counts'].items(), start=r2+1):
        write_kv(ws1, idx, vtype, str(cnt))
        ws1.row_dimensions[idx].height = 18

    ws1.column_dimensions['A'].width = 32
    for col in ['B','C','D','E','F','G','H']:
        ws1.column_dimensions[col].width = 16

    # ════════════════════════════════════════════════════════════
    # SHEET 2 — DETAILED FINDINGS
    # ════════════════════════════════════════════════════════════
    ws2 = wb.create_sheet('🔍 Detailed Findings')
    ws2.sheet_view.showGridLines = False
    ws2.freeze_panes = 'A3'

    DETAIL_COLS = [
        ('Repository Name',    18),
        ('File Name',          38),
        ('Line #',             8),
        ('Type',               26),
        ('Vulnerability',      38),
        ('Severity',           12),
        ('Score',              8),
        ('Code Snippet',       42),
        ('AI Explanation',     52),
        ('Risk Impact',        46),
        ('Recommended Fix',    52),
        ('Secure Code',        55),
        ('Reference',          38),
    ]
    DF_COLS = [
        'Repository Name', 'File Name', 'Line Number',
        'Vulnerability Type', 'Vulnerability Name', 'Severity', 'Risk Score',
        'Code Snippet', 'AI Explanation', 'Risk Impact',
        'Recommended Fix', 'Secure Code Example', 'Reference Link'
    ]

    # Sheet title row
    n_cols = len(DETAIL_COLS)
    ws2.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
    c = ws2.cell(row=1, column=1,
                 value='🔍  DETAILED VULNERABILITY FINDINGS WITH AI REMEDIATION')
    c.font      = cell_font(bold=True, size=13, color='FFFFFF')
    c.fill      = hdr_fill('0F172A')
    c.alignment = Alignment(horizontal='center', vertical='center')
    ws2.row_dimensions[1].height = 30

    # Column headers
    for ci, (hdr, w) in enumerate(DETAIL_COLS, start=1):
        c = ws2.cell(row=2, column=ci, value=hdr)
        c.font      = cell_font(bold=True, size=10, color='FFFFFF')
        c.fill      = hdr_fill('1E3A5F')
        c.border    = bdr
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws2.column_dimensions[get_column_letter(ci)].width = w
    ws2.row_dimensions[2].height = 26

    # Data rows
    for ri, row in enumerate(df[DF_COLS].itertuples(index=False), start=3):
        sev    = row[5]   # Severity
        alt_bg = 'F8FAFC' if ri % 2 == 0 else 'FFFFFF'

        for ci, val in enumerate(row, start=1):
            c = ws2.cell(row=ri, column=ci,
                         value=(str(val) if val is not None else ''))
            c.border    = bdr
            c.alignment = Alignment(vertical='top', wrap_text=True)
            c.font      = Font(size=9, name='Arial')

            if ci == 6:   # Severity
                c.fill      = hdr_fill(SEVERITY_BG_HEX.get(sev, 'FFFFFF'))
                c.font      = cell_font(bold=True, size=9,
                                        color=SEVERITY_FG_HEX.get(sev, '000000'))
                c.alignment = Alignment(horizontal='center', vertical='top')
            elif ci == 7:  # Score
                c.fill      = hdr_fill(SEVERITY_BG_HEX.get(sev, 'FFFFFF'))
                c.font      = cell_font(bold=True, size=9,
                                        color=SEVERITY_FG_HEX.get(sev, '000000'))
                c.alignment = Alignment(horizontal='center', vertical='top')
            elif ci == 12:  # Secure Code — monospace
                c.fill = hdr_fill('F0FDF4')
                c.font = Font(size=8, name='Courier New', color='15803D')
            else:
                c.fill = hdr_fill(alt_bg)

        ws2.row_dimensions[ri].height = 72

    wb.save(filepath)
    print(f'\n✅ Excel report saved → {filepath}')
    print(f'   Sheets : {[s.title for s in wb.worksheets]}')
    print(f'   Rows   : {len(df)} vulnerability findings')


# ── Export ───────────────────────────────────────────────────────────
if not df.empty:
    build_excel_report(df, risk_report, OUTPUT_XLSX)
else:
    print('ℹ️  No vulnerabilities found — Excel report not generated.')

---
## ✅ 11. Final Execution Output

In [ ]:
def print_final_report(df: pd.DataFrame, risk_report: dict, max_show: int = 15):
    """Rich terminal-style vulnerability report + final dashboard."""
    EMOJI = {'Critical': '🔴', 'High': '🟠', 'Medium': '🟡', 'Low': '🟢'}
    meta  = risk_report['meta']

    print('\n' + '═'*72)
    print('  🔍  VULNERABILITY FINDINGS  (top results)')
    print('═'*72)

    if df.empty:
        print('  ✅  No vulnerabilities detected in this repository.')
    else:
        for i, row in df.head(max_show).iterrows():
            e = EMOJI.get(row['Severity'], '⚪')
            print(f"""
#{i+1:02d}  {e} [{row['Severity'].upper()}]  {row['Vulnerability Name']}
    📁  File       : {row['File Name']}  (Line {row['Line Number']})
    🏷️   Type       : {row['Vulnerability Type']}
    💬  Explanation: {str(row['AI Explanation'])[:160]}...
    ⚡  Risk Impact : {str(row['Risk Impact'])[:140]}...
    🔧  Fix         : {str(row['Recommended Fix'])[:150]}
    🔗  Ref         : {row['Reference Link']}
    {'─'*68}""")

        if len(df) > max_show:
            print(f'\n  ... and {len(df)-max_show} more findings in the Excel report.')

    # ── Final dashboard ──────────────────────────────────────────────
    sc  = risk_report['sev_counts']
    cls = risk_report['classification']
    CLS_EMOJI = {
        'Safe': '✅', 'Moderate Risk': '⚠️ ',
        'High Risk': '🔶', 'Critical Risk': '🔴'
    }

    print('\n\n' + '╔' + '═'*70 + '╗')
    print('║' + '  🔐  FINAL SECURITY SCAN DASHBOARD'.center(70) + '║')
    print('╠' + '═'*70 + '╣')
    print(f'║  Repository      : {meta["full_name"]:<51} ║')
    print(f'║  Scanned At      : {meta["scanned_at"]:<51} ║')
    print(f'║  Primary Language: {meta["language"]:<51} ║')
    print(f'║  AI Model        : {HF_MODEL:<51} ║')
    print('╠' + '═'*70 + '╣')
    print(f'║  Total Findings  : {len(df):<51} ║')
    for sev in ['Critical', 'High', 'Medium', 'Low']:
        n = sc.get(sev, 0)
        print(f'║    {sev:<12}: {n:<53} ║')
    print('╠' + '═'*70 + '╣')
    print(f'║  Total Risk Score: {risk_report["total_score"]:<51} ║')
    cls_str = f'{CLS_EMOJI.get(cls,"")} {cls}'
    print(f'║  Classification  : {cls_str:<51} ║')
    print('╠' + '═'*70 + '╣')
    print(f'║  📄 Excel Report : {OUTPUT_XLSX:<51} ║')
    print(f'║  📊 Charts       : security_dashboard.png{" "*27} ║')
    print('╚' + '═'*70 + '╝')

    print('\n📋 RECOMMENDED NEXT STEPS:')
    print('  1. Open the Excel report → "📊 Executive Summary" for the risk overview')
    print('  2. Work through "🔍 Detailed Findings" sorted by Severity')
    if sc.get('Critical', 0):
        print('  3. 🚨 CRITICAL issues found — rotate any exposed credentials IMMEDIATELY')
    if sc.get('High', 0):
        print('  4. Address HIGH findings before the next production deployment')
    print('  5. Add GitHub Actions secret-scanning + OWASP Dependency-Check to CI/CD')
    print('  6. Use Spring Boot Vault integration to eliminate hardcoded secrets')


print_final_report(df, risk_report)

---

## 📚 Quick Reference

### 🚀 Running the Notebook

1. Get a **free Hugging Face token** → https://huggingface.co/settings/tokens  
   _(New token → Role: Read → copy the `hf_...` string)_
2. Optionally get a **GitHub token** → https://github.com/settings/tokens  
   _(New classic token → tick `repo` → copy `ghp_...`)_
3. Paste both tokens in **Section 4** (`GITHUB_TOKEN`, `HF_TOKEN`)
4. **Kernel → Restart & Run All**

---

### ⚙️ Configuration

| Variable | Default | Description |
|---|---|---|
| `TARGET_REPO` | `mayureshnadkar-beep/spring-ai-ollama` | GitHub `owner/repo` |
| `HF_MODEL` | `google/flan-t5-large` | Hugging Face model ID |
| `MAX_FILES` | `120` | Max files to traverse |
| `HF_CALL_DELAY` | `1.5` | Seconds between HF API calls |
| `OUTPUT_XLSX` | `github_security_vulnerability_report.xlsx` | Export filename |

---

### 📊 Risk Classification

| Total Score | Classification | Meaning |
|---|---|---|
| 0 – 5 | ✅ Safe | No significant issues |
| 6 – 15 | ⚠️ Moderate Risk | Low-priority fixes needed |
| 16 – 30 | 🔶 High Risk | Fix before next release |
| 31+ | 🔴 Critical Risk | Immediate action required |

### 🤖 AI Fallback
If no HF token is set, the notebook uses a built-in expert rule database covering all 17 Java/Spring vulnerability patterns with hand-crafted explanations, impact assessments, remediation steps and secure code examples — **no internet required**.